# 02 Exploración: Inside Airbnb (Euskadi, snapshot 2026-06-30)

**Objetivo:** decidir qué columnas y qué reglas de limpieza llegan a silver, y qué se puede medir con estos datos.
**Datos:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), licencia CC BY 4.0. Bronze en `data/bronze/<ciudad>/<fecha>/`.
**Manual:** `docs/manual.md` recoge el porqué de cada decisión. Este notebook es la evidencia.
**Ejecución:** de arriba abajo con *Run All Cells* (el kernel no recuerda variables al reabrir).

## 2a Radiografía de `listings`
**Qué:** cargo `listings` y miro forma, tipos de columna y columnas 100 % vacías.
**Por qué:** antes de limpiar hay que saber qué hay. Una columna vacía se tipa como `float64`, así que el tipo no dice qué es.
**Resultado:** 6.248 anuncios, 90 columnas, 12 completamente vacías (`host_since`, `neighbourhood`, `instant_bookable`...).

In [1]:
import pandas as pd
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config/snapshots.yaml").exists())
f = RAIZ / "data/bronze/euskadi/2026-06-30/listings.csv.gz"
print("RAIZ:", RAIZ)
df = pd.read_csv(f)

print(df.shape)
print(df.dtypes.value_counts())
vacias = df.columns[df.isna().all()]
print(len(vacias), "columnas 100% vacías:", list(vacias))

RAIZ: C:\Users\PCUser\revenue-intelligence-airbnb
(6248, 90)
float64    42
str        30
int64      18
Name: count, dtype: int64
12 columnas 100% vacías: ['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'calendar_updated', 'instant_bookable']


## 2b La trampa del `price`
**Qué:** `price` viene como texto (`"$483.00"`). Lo convierto a número y miro nulos, comas de miles y extremos.
**Por qué:** es la columna clave del ADR y del RevPAR.
**Resultado:** 572 nulos (9,2 %), 87 precios con coma de miles, máximo 9999. En `crosstab`, los nulos NO se concentran en `has_availability = f`.

In [2]:
p = df["price"]
print("nulos:", p.isna().sum(), f"({p.isna().mean():.1%})")
print("muestra:", p.dropna().sample(8, random_state=1).tolist())
print("con coma de miles:", p.str.contains(",", na=False).sum())

num = pd.to_numeric(p.str.replace(r"[$,]", "", regex=True), errors="coerce")
print(num.describe())
print("5 más caros:", num.sort_values().tail(5).tolist())
print("precio 0:", (num == 0).sum())

# ¿Los nulos son anuncios inactivos?
print(pd.crosstab(p.isna(), df["has_availability"], margins=True))

nulos: 572 (9.2%)
muestra: ['$150.50', '$103.00', '$163.00', '$70.00', '$192.75', '$350.00', '$261.50', '$68.50']
con coma de miles: 87
count    5676.000000
mean      264.625157
std       276.032078
min         4.840000
25%       130.500000
50%       196.000000
75%       315.542500
max      9999.000000
Name: price, dtype: float64
5 más caros: [nan, nan, nan, nan, nan]
precio 0: 0
has_availability  f     t   All
price                          
False             4  5672  5676
True              2   544   546
All               6  6216  6222


## 2c Extremos y nulos de precio
**Qué:** miro los precios más altos y más bajos, y los nulos según su disponibilidad.
**Por qué:** decidir si 9999 y los precios bajos son errores o casos reales.
**Resultado:** solo 1 anuncio a 9999. 87 anuncios >= 1000 parecen reales (casas grandes con reseñas). 15 anuncios < 20. 26 filas sin `has_availability`.

In [3]:
print("más caros:", num.dropna().sort_values().tail(8).tolist())
print("≥1000:", (num >= 1000).sum(), "| ==9999:", (num == 9999).sum(), "| <20:", (num < 20).sum())
print(df.loc[num >= 1000, ["room_type", "accommodates", "availability_365", "number_of_reviews_ltm"]].head(8))

print("\nhas_availability nulo:", df["has_availability"].isna().sum())
nulos = df[p.isna()]
print(nulos[["availability_30", "availability_365", "number_of_reviews_ltm", "price_quote_price_per_night"]].describe().T)

más caros: [2856.42, 2919.5, 3301.0, 3424.0, 3483.0, 4049.0, 4222.0, 9999.0]
≥1000: 87 | ==9999: 1 | <20: 15
           room_type  accommodates  availability_365  number_of_reviews_ltm
95   Entire home/apt             8               274                      9
142  Entire home/apt             8               171                      0
170  Entire home/apt            10               278                     10
200  Entire home/apt             8               286                     24
259  Entire home/apt            15                63                      2
347  Entire home/apt             6               140                      5
418  Entire home/apt             8               276                     38
427  Entire home/apt             6               342                      0

has_availability nulo: 26
                             count       mean         std  min  25%  50%  \
availability_30              572.0   4.641608    8.279438  0.0  0.0  0.0   
availability_365            

## 2d Dos tipos de nulos y los precios bajos
**Qué:** separo los nulos de precio por `availability_365` y miro los 15 anuncios baratos.
**Por qué:** un nulo con 0 días libres es natural; uno con muchos días libres es un problema de captura.
**Resultado:** 281 nulos con 0 días libres y 230 con >= 30. Los 15 baratos son habitaciones privadas con `minimum_nights` >= 31: larga estancia.

In [4]:
n = df[p.isna()]
print("nulos con availability_365 == 0 :", (n["availability_365"] == 0).sum())
print("nulos con availability_365 >= 30:", (n["availability_365"] >= 30).sum())
print("price_quote_price_per_night informado en todo el fichero:", df["price_quote_price_per_night"].notna().sum())

baratos = (df.loc[num < 20, ["room_type", "accommodates", "minimum_nights", "availability_365"]]
             .assign(precio=num[num < 20]).sort_values("precio"))
print(baratos.to_string())

nulos con availability_365 == 0 : 281
nulos con availability_365 >= 30: 230
price_quote_price_per_night informado en todo el fichero: 5676
         room_type  accommodates  minimum_nights  availability_365  precio
2447  Private room             1            90.0               296    4.84
1103  Private room             1            90.0               243    6.24
1099  Private room             1            90.0               182    6.49
1699  Private room             1            90.0               158    6.79
2787  Private room             1            90.0               167    6.79
1939  Private room             2            90.0               274    7.69
1576  Private room             2            90.0               305    8.07
4832  Private room             2            91.0               364    9.60
4809  Private room             3            91.0               364   12.96
5822  Private room             1            31.0               122   12.96
5728  Private room             1    

## 2e Validar el parseo y medir la larga estancia
**Qué:** comparo mi `price` numérico con `price_quote_price_per_night` y cuento anuncios por `minimum_nights`.
**Por qué:** validar que la conversión no pierde datos y decidir el umbral de larga estancia.
**Resultado:** coinciden al 100 %. 264 anuncios con >= 28 noches; su mediana de precio es 95,52 frente a 199,80 del resto.

In [5]:
q = df["price_quote_price_per_night"]
ok = num.notna()
print("price == price_quote (±0.01):", ((num[ok] - q[ok]).abs() < 0.01).mean())

print(df["minimum_nights"].describe())
for u in [1, 2, 3, 7, 28, 30, 90]:
    print(f"minimum_nights >= {u:>2}:", (df["minimum_nights"] >= u).sum())

print(num.groupby(df["minimum_nights"] >= 28).agg(["count", "median", "mean"]))

price == price_quote (±0.01): 1.0
count    6246.000000
mean        3.024976
std         7.162174
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       100.000000
Name: minimum_nights, dtype: float64
minimum_nights >=  1: 6246
minimum_nights >=  2: 2916
minimum_nights >=  3: 846
minimum_nights >=  7: 325
minimum_nights >= 28: 264
minimum_nights >= 30: 263
minimum_nights >= 90: 12
                count  median        mean
minimum_nights                           
False            5457  199.80  268.844026
True              219   95.52  159.500183


## 2f Radiografía de `calendar`
**Qué:** cargo el calendario de este snapshot y miro columnas, fechas, `available` e integridad con `listings`.
**Por qué:** es la tabla grande (anuncio x noche) y base de la ocupación y el pickup.
**Resultado:** 2.281.585 filas, 5 columnas y **sin precio**. 50,4 % `t` y 49,6 % `f`. 3 anuncios de `calendar` no están en `listings`.

In [6]:
f = RAIZ / "data/bronze/euskadi/2026-06-30/calendar.csv.gz"
cal = pd.read_csv(f)

print(cal.shape, "| memoria MB:", round(cal.memory_usage(deep=True).sum() / 1e6))
print(cal.dtypes)
print(cal.head(4))
print("fechas:", cal["date"].min(), "->", cal["date"].max())
print(cal["available"].value_counts(normalize=True))

por_listing = cal.groupby("listing_id").size()
print("listings en calendar:", por_listing.size, "| filas por listing (min/max):", por_listing.min(), por_listing.max())
print("en calendar y no en listings:", len(set(cal["listing_id"]) - set(df["id"])))
print("en listings y no en calendar:", len(set(df["id"]) - set(cal["listing_id"])))

(2281585, 5) | memoria MB: 116
listing_id        int64
date                str
available           str
minimum_nights    int64
maximum_nights    int64
dtype: object
   listing_id        date available  minimum_nights  maximum_nights
0      132068  2026-07-01         f               1              63
1      132068  2026-07-02         f               2              63
2      132068  2026-07-03         f               2              63
3      132068  2026-07-04         f               2              63
fechas: 2026-06-30 -> 2027-07-02
available
t    0.503979
f    0.496021
Name: proportion, dtype: float64
listings en calendar: 6251 | filas por listing (min/max): 335 365
en calendar y no en listings: 3
en listings y no en calendar: 0


## 2g Huérfanos y ventanas de fechas
**Qué:** miro los 3 anuncios que solo están en `calendar` y la fecha de inicio y fin de cada anuncio.
**Por qué:** entender la integridad y si el snapshot tiene una fecha de captura única.
**Resultado:** los 3 huérfanos son anuncios nuevos. La captura dura ~4 días (inicio entre 06-30 y 07-03), así que la fecha del snapshot es nominal.

In [7]:
huerfanos = sorted(set(cal["listing_id"]) - set(df["id"]))
print("huérfanos:", huerfanos)
print(cal[cal["listing_id"].isin(huerfanos)].groupby("listing_id")
        .agg(filas=("date", "size"), desde=("date", "min"), hasta=("date", "max")))

print("\ninicio de ventana por anuncio:")
print(cal.groupby("listing_id")["date"].min().value_counts().head(5))
print("\nfin de ventana por anuncio:")
print(cal.groupby("listing_id")["date"].max().value_counts().head(5))

huérfanos: [1472367620161798363, 1472368242300662346, 1506532257136321938]
                     filas       desde       hasta
listing_id                                        
1472367620161798363    365  2026-07-02  2027-07-01
1472368242300662346    365  2026-07-02  2027-07-01
1506532257136321938    365  2026-07-03  2027-07-02

inicio de ventana por anuncio:
date
2026-07-01    4804
2026-06-30     931
2026-07-02     399
2026-07-03     117
Name: count, dtype: int64

fin de ventana por anuncio:
date
2027-06-30    4803
2027-06-29     931
2027-07-01     399
2027-07-02     117
2027-05-31       1
Name: count, dtype: int64


## 2h ¿Trae precio el `calendar` antiguo?
**Qué:** cargo un `calendar` antiguo (Euskadi 2025-09-29), que sí tiene columnas `price` y `adjusted_price`.
**Por qué:** una columna que existe no tiene por qué tener datos.
**Resultado:** ambas están 100 % vacías. Verificado en los 9 snapshots que las tienen. El único precio es el de `listings`.

In [9]:
cal_old = pd.read_csv(RAIZ / "data/bronze/euskadi/2025-09-29/calendar.csv.gz")
print(cal_old.shape)
print(cal_old.dtypes)
print(cal_old.head(4))

for c in ["price", "adjusted_price"]:
    if c in cal_old.columns:
        print(c, "nulos:", cal_old[c].isna().mean().round(3))

print(cal_old.groupby("available")["price"].apply(lambda s: s.notna().mean()))

(2311910, 7)
listing_id          int64
date                  str
available             str
price             float64
adjusted_price    float64
minimum_nights      int64
maximum_nights      int64
dtype: object
   listing_id        date available  price  adjusted_price  minimum_nights  \
0     8937924  2025-09-30         f    NaN             NaN               1   
1     8937924  2025-10-01         t    NaN             NaN               1   
2     8937924  2025-10-02         t    NaN             NaN               1   
3     8937924  2025-10-03         f    NaN             NaN               1   

   maximum_nights  
0            1125  
1            1125  
2            1125  
3            1125  
price nulos: 1.0
adjusted_price nulos: 1.0
available
f    0.0
t    0.0
Name: price, dtype: float64


## 2i Prototipo de RevPAR aproximado
**Qué:** agrego `calendar` a un valor por anuncio (proporción de noches `f`), lo uno con `listings` y calculo `precio x ocupación` por zona.
**Por qué:** comprobar que con estos datos se puede construir la métrica antes de diseñar silver.
**Resultado:** de 6.248 anuncios quedan 5.454 en la base. Solo 25 de 206 zonas tienen >= 30 anuncios. La ocupación es un límite superior.

In [10]:
# Ocupación aprox. por anuncio = proporción de noches 'f' en su ventana
occ = (cal.assign(no_disp=cal["available"].eq("f"))
          .groupby("listing_id")["no_disp"].mean().rename("occ_aprox"))

x = df.assign(price_num=num).merge(occ, left_on="id", right_index=True, how="left")
x["revpar_aprox"] = x["price_num"] * x["occ_aprox"]

# Solo corta estancia, con precio y sin el centinela
base = x[(x["minimum_nights"] < 28) & x["price_num"].notna() & (x["price_num"] < 9999)]
print("anuncios totales:", len(x), "| en la base:", len(base))

res = (base.groupby("neighbourhood_cleansed")
           .agg(anuncios=("id", "size"), precio_mediano=("price_num", "median"),
                occ_mediana=("occ_aprox", "median"), revpar_mediano=("revpar_aprox", "median"))
           .query("anuncios >= 30").sort_values("revpar_mediano", ascending=False))
print(res.head(10).round(2))

anuncios totales: 6248 | en la base: 5454
                         anuncios  precio_mediano  occ_mediana  revpar_mediano
neighbourhood_cleansed                                                        
Zarautz                       143          274.17         0.67          158.99
Donostia-San SebastiÃ¡n      1316          347.00         0.52          145.68
Getaria                        43          224.00         0.72          135.57
Hondarribia                   114          239.90         0.58          130.99
Gorliz                         31          188.50         0.48          109.64
Gautegiz Arteaga               43          241.67         0.35          107.65
Getxo                         123          214.67         0.60          101.92
Bakio                          56          208.82         0.63           99.46
Plentzia                       30          184.17         0.61           89.50
Ibarrangelu                    31          205.50         0.56           78.54


## 3.1 Elegir 39 columnas y declarar los tipos
**Qué:** cargo solo 39 columnas con el tipo declarado a mano, en vez de dejar que pandas lo adivine.
**Por qué:** los tipos no deben cambiar entre snapshots, y no se arrastran datos personales ni columnas inútiles.
**Resultado:** (6.248, 39): 15 `Int64`, 12 `float64`, 8 `string` y 4 fechas.

In [11]:
COLS_ENTERAS = ["id", "host_id", "accommodates", "availability_30", "availability_60",
    "availability_90", "availability_365", "number_of_reviews", "number_of_reviews_ltm",
    "number_of_reviews_l30d", "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes", "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms", "estimated_occupancy_l365d"]
COLS_DECIMALES = ["latitude", "longitude", "bathrooms", "bedrooms", "beds", "minimum_nights",
    "maximum_nights", "review_scores_rating", "review_scores_location", "review_scores_value",
    "reviews_per_month", "estimated_revenue_l365d"]
COLS_FECHA = ["last_scraped", "calendar_last_scraped", "first_review", "last_review"]
COLS_TEXTO = ["neighbourhood_cleansed", "neighbourhood_group_cleansed", "property_type",
    "room_type", "price", "license", "has_availability", "host_is_superhost"]

schema = {**{c: "Int64" for c in COLS_ENTERAS},
          **{c: "float64" for c in COLS_DECIMALES},
          **{c: "string" for c in COLS_TEXTO}}

lst = pd.read_csv(RAIZ / "data/bronze/euskadi/2026-06-30/listings.csv.gz",
                  usecols=list(schema) + COLS_FECHA, dtype=schema, parse_dates=COLS_FECHA)

print(lst.shape)
print(lst.dtypes.value_counts())
print(lst.isna().mean().round(3).sort_values(ascending=False).head(8))

(6248, 39)
Int64             15
float64           12
string             8
datetime64[us]     4
Name: count, dtype: int64
bathrooms               0.155
bedrooms                0.149
license                 0.118
beds                    0.115
last_review             0.101
first_review            0.101
review_scores_rating    0.101
review_scores_value     0.101
dtype: float64


## 3.2 Reparar acentos, convertir el precio y marcar casos
**Qué:** corrijo los acentos de las zonas, convierto `price` a número y creo tres flags.
**Por qué:** bronze guarda el dato tal cual; silver es donde se limpia, sin borrar nada.
**Resultado esperado:** 0 acentos rotos, 572 sin precio, 1 outlier, 264 larga estancia.

In [12]:
def reparar_texto(s):
    """Texto UTF-8 leído como latin-1: 'SebastiÃ¡n' -> 'Sebastián'."""
    if pd.isna(s) or "Ã" not in s:
        return s
    try:
        return s.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

silver = lst.copy()
for c in ["neighbourhood_cleansed", "neighbourhood_group_cleansed"]:
    silver[c] = silver[c].map(reparar_texto).astype("string")

LARGA_ESTANCIA = 28
silver["price_num"] = pd.to_numeric(silver["price"].str.replace(r"[$,]", "", regex=True), errors="coerce")
silver["price_missing"] = silver["price_num"].isna()
silver["price_outlier"] = silver["price_num"].eq(9999)
silver["is_long_stay"] = silver["minimum_nights"].ge(LARGA_ESTANCIA)

print(silver["neighbourhood_group_cleansed"].drop_duplicates().tolist())
print("con 'Ã' restantes:", silver["neighbourhood_cleansed"].str.contains("Ã", na=False).sum())
print(silver[["price_missing", "price_outlier", "is_long_stay"]].sum())
print(silver["price_num"].describe().round(2))

['Guipúzcoa', 'Vizcaya', 'Álava']
con 'Ã' restantes: 0
price_missing    572
price_outlier      1
is_long_stay     264
dtype: Int64
count    5676.0
mean     264.63
std      276.03
min        4.84
25%       130.5
50%       196.0
75%      315.54
max      9999.0
Name: price_num, dtype: Float64


---
# Evidencias de la exploración
Comprobaciones sobre **los 23 snapshots** que respaldan las decisiones del manual (`docs/manual.md`). Se ejecutan de arriba abajo; la primera celda de código del cuaderno define `RAIZ`.

## E1 Deriva de esquema en `listings`
**Qué:** leo solo la cabecera de cada `listings` y comparo las columnas entre los 23 snapshots.
**Por qué:** si la fuente cambia columnas, silver no puede asumir un esquema fijo.
**Resultado esperado:** 23 snapshots, 79 columnas comunes, snapshots con 79, 85 o 90 columnas, y `price_quote_price_per_night` ausente en 15.

In [14]:
import glob

rutas = sorted(glob.glob(str(RAIZ / "data/bronze/*/*/listings.csv.gz")))
cols = {r: set(pd.read_csv(r, nrows=1).columns) for r in rutas}
comunes = set.intersection(*cols.values())

print("snapshots:", len(cols), "| columnas en todos:", len(comunes))
print("columnas por snapshot:", sorted({len(v) for v in cols.values()}))
print("price_quote_price_per_night ausente en:", sum("price_quote_price_per_night" not in v for v in cols.values()), "snapshots")

snapshots: 23 | columnas en todos: 79
columnas por snapshot: [79, 85, 90]
price_quote_price_per_night ausente en: 15 snapshots


## E2 El `price` del `calendar` antiguo está vacío
**Qué:** en los snapshots cuyo `calendar` tiene columna `price`, mido qué porcentaje de filas la trae informada.
**Por qué:** que una columna exista no significa que tenga datos; define si hay precio por noche.
**Resultado esperado:** 9 snapshots (6 de Barcelona y 3 de Euskadi) con 0 % de `price` y `adjusted_price` informado. Tarda un par de minutos (~50 millones de filas).

In [15]:
filas = []
for r in sorted(glob.glob(str(RAIZ / "data/bronze/*/*/calendar.csv.gz"))):
    if "price" not in pd.read_csv(r, nrows=1).columns:
        continue
    d = pd.read_csv(r, usecols=["price", "adjusted_price"])
    ciudad, fecha = Path(r).parts[-3:-1]
    filas.append((ciudad, fecha, len(d), d["price"].notna().mean(), d["adjusted_price"].notna().mean()))

pd.DataFrame(filas, columns=["ciudad", "fecha", "filas", "price_informado", "adjusted_informado"])

,ciudad,fecha,filas,price_informado,adjusted_informado
0,barcelona,2025-08-10,7053632,0.0,0.0
1,barcelona,2025-09-14,7084654,0.0,0.0
2,barcelona,2025-10-17,5502377,0.0,0.0
3,barcelona,2025-11-17,6545937,0.0,0.0
4,barcelona,2025-12-14,6634623,0.0,0.0
5,barcelona,2026-01-18,6241135,0.0,0.0
6,euskadi,2025-07-30,2374690,0.0,0.0
7,euskadi,2025-08-31,2370310,0.0,0.0
8,euskadi,2025-09-29,2311910,0.0,0.0


## E3 Acentos rotos (mojibake) y estrategia de reparación
**Qué:** cuento los nombres de zona con `Ã` en Euskadi y Barcelona, y pruebo dos formas de repararlos sobre **todos** los valores afectados.
**Por qué:** una reparación probada con un solo ejemplo puede fallar en otros (`Álava` falla con `cp1252`).
**Resultado esperado:** Euskadi (1692, 6248), Barcelona (0, 15293). 22 valores afectados: `cp1252` repara 20 y `latin-1` repara los 22.

In [16]:
def contar_mojibake(ruta):
    z = pd.read_csv(ruta, usecols=["neighbourhood_cleansed"])["neighbourhood_cleansed"]
    return int(z.str.contains("Ã", na=False).sum()), len(z)

eus = RAIZ / "data/bronze/euskadi/2026-06-30/listings.csv.gz"
print("Euskadi  :", contar_mojibake(eus))
print("Barcelona:", contar_mojibake(RAIZ / "data/bronze/barcelona/2026-06-24/listings.csv.gz"))

z = pd.read_csv(eus, usecols=["neighbourhood_group_cleansed", "neighbourhood_cleansed"])
valores = pd.unique(pd.concat([z["neighbourhood_group_cleansed"], z["neighbourhood_cleansed"]]).dropna())
malos = [s for s in valores if "Ã" in s]

def reparables(codificacion):
    ok = 0
    for s in malos:
        try:
            s.encode(codificacion).decode("utf-8")
            ok += 1
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return ok

print("valores afectados:", len(malos), "| cp1252:", reparables("cp1252"), "| latin-1:", reparables("latin-1"))

Euskadi  : (1692, 6248)
Barcelona: (0, 15293)
valores afectados: 22 | cp1252: 20 | latin-1: 22


## E4 Jerarquía de zonas: grupo y zona
**Qué:** cuento cuántas zonas (`neighbourhood_cleansed`) hay dentro de cada grupo (`neighbourhood_group_cleansed`) en Euskadi y Barcelona.
**Por qué:** en Euskadi las "zonas" son municipios, no barrios; el grupo da un segundo nivel para comparar (`dim_zona` en gold).
**Resultado esperado:** Euskadi: Álava 44, Guipúzcoa 62, Vizcaya 101. Barcelona: 10 distritos y 69 barrios. Usa `silver` de la celda 3.2.

In [17]:
print(silver.groupby("neighbourhood_group_cleansed")["neighbourhood_cleansed"].nunique())

b = pd.read_csv(RAIZ / "data/bronze/barcelona/2026-06-24/listings.csv.gz",
                usecols=["neighbourhood_group_cleansed", "neighbourhood_cleansed"])
print("Barcelona: distritos", b["neighbourhood_group_cleansed"].nunique(), "| barrios", b["neighbourhood_cleansed"].nunique())

neighbourhood_group_cleansed
Guipúzcoa     62
Vizcaya      101
Álava         44
Name: neighbourhood_cleansed, dtype: int64
Barcelona: distritos 10 | barrios 69
